# Phase 7: MIAM Fortran Bindings

Verifies the MIAM Fortran bindings in the MUSICA library by **running the
Fortran integration test directly** and parsing its stdout.

The test (`fortran/test/integration/test_miam_cloud_chemistry.F90`):

1. Creates a MIAM-enabled solver from `configs/v1/cam_cloud_chemistry/config.json`
2. Verifies the state exposes the expected cloud species
3. Sets self-consistent equilibrium ICs and integrates 1800 s with the
   DAE4 Rosenbrock solver, asserting kinetic SO4 production

Phase 7 passes only if all three Fortran tests pass *and* the Fortran
SO4-production result agrees with the Python equivalent
(`python/test/integration/test_miam_cloud_chemistry.py::TestKineticsValidation::test_so4_increases`)
to within tolerance.

This phase exercises code in the MUSICA repo (the parent of this MPAS-Model
checkout); it does not require any pre-staged data files.

In [ ]:
import re
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Notebook lives at MPAS-Model/verification/; MUSICA repo is two levels up.
NOTEBOOK_DIR = Path.cwd()
MUSICA_DIR = (NOTEBOOK_DIR / ".." / "..").resolve()
print(f"MUSICA repo:    {MUSICA_DIR}")

# Locate a built test binary. setup_and_run.sh builds inside a container, but
# the MUSICA Fortran test lives in build-miam/ on the host.
candidates = [
    MUSICA_DIR / "build-miam" / "test_miam_cloud_chemistry",
    MUSICA_DIR / "build-container" / "test_miam_cloud_chemistry",
    MUSICA_DIR / "build" / "test_miam_cloud_chemistry",
]
binary = next((p for p in candidates if p.exists() and p.is_file()), None)
if binary is None:
    raise FileNotFoundError(
        "Fortran MIAM integration test binary not found. Build MUSICA first:\n"
        "    cmake -S . -B build-miam -DMUSICA_BUILD_FORTRAN_INTERFACE=ON \\\n"
        "          -DMUSICA_ENABLE_TESTS=ON\n"
        "    cmake --build build-miam --target test_miam_cloud_chemistry"
    )
print(f"Test binary:    {binary.relative_to(MUSICA_DIR)}")

## 1. Run the Fortran integration test

Run the binary from the MUSICA repo root (so the relative `configs/...`
path inside the test resolves correctly) and capture stdout.

In [ ]:
proc = subprocess.run(
    [str(binary)],
    cwd=MUSICA_DIR,
    capture_output=True,
    text=True,
    timeout=120,
)

print("---- Fortran test stdout ----")
print(proc.stdout)
if proc.stderr.strip():
    print("---- stderr ----")
    print(proc.stderr)
print(f"---- exit code: {proc.returncode} ----")

assert proc.returncode == 0, (
    f"Fortran MIAM integration test failed (exit code {proc.returncode}). "
    "Inspect the stdout above; an `[MUSICA ERROR ... Assertion failed]` line "
    "indicates which check tripped."
)

# All three sub-tests must report PASSED.
passed_subtests = re.findall(r"^\s*PASSED\s*$", proc.stdout, flags=re.MULTILINE)
assert len(passed_subtests) >= 3, (
    f"Expected 3 sub-tests to PASS; got {len(passed_subtests)}. "
    "Stdout above shows which one was missing."
)
print(f"Sub-tests passed: {len(passed_subtests)}")

## 2. Cross-validate Fortran vs Python SO4 production

Parse the `SO4 initial`/`SO4 final` values from the Fortran stdout and
compare against the Python equivalent. Both runs use the same config,
same DAE4 solver, same naive ICs, and integrate to t = 1800 s; their
final SO4 should agree to leading-order.

In [ ]:
m = re.search(
    r"SO4\s+initial:\s*([0-9eE+\-.]+)\s+final:\s*([0-9eE+\-.]+)",
    proc.stdout,
)
assert m, "Could not find 'SO4 initial: ... final: ...' in Fortran stdout."

so4_initial_f = float(m.group(1))
so4_final_f = float(m.group(2))
delta_f = so4_final_f - so4_initial_f

print(f"Fortran  SO4 initial: {so4_initial_f:.15e}")
print(f"Fortran  SO4 final:   {so4_final_f:.15e}")
print(f"Fortran  Delta SO4:   {delta_f:.6e}")

assert delta_f > 0, f"Fortran reported no SO4 production (delta = {delta_f:.3e})."

In [ ]:
# Run the Python equivalent in-process and compare.
import importlib
import math
import sys

PY_TEST_DIR = MUSICA_DIR / "python" / "test" / "integration"
if str(PY_TEST_DIR) not in sys.path:
    sys.path.insert(0, str(PY_TEST_DIR))

mod = importlib.import_module("test_miam_cloud_chemistry")

from musica.micm import MICM, SolverType

mechanism = mod._create_gas_mechanism()
model = mod._create_kinetics_model()
micm = MICM(
    mechanism=mechanism,
    solver_type=SolverType.rosenbrock_dae4_standard_order,
    external_models=[model],
)
state = micm.create_state()
model.set_default_parameters(state)
state.set_conditions(temperatures=mod.T_INIT, pressures=mod.P_INIT)
state.set_concentrations(mod._naive_initial_conditions())
mod._integrate(micm, state, target_time=1800.0, dt_init=0.001)

concs = state.get_concentrations()
so4_initial_p = mod.SO4MM0
so4_final_p = concs["CLOUD.AQUEOUS.SO4mm"][0]
delta_p = so4_final_p - so4_initial_p

print(f"Python   SO4 initial: {so4_initial_p:.15e}")
print(f"Python   SO4 final:   {so4_final_p:.15e}")
print(f"Python   Delta SO4:   {delta_p:.6e}")

In [ ]:
# Both implementations should agree. Tolerance is loose because Fortran
# uses a different time-step schedule than the Python helper, and Delta is
# small (~1e-14 mol/m^3) -- relative scatter at the 50% level is normal.
abs_err = abs(so4_final_f - so4_final_p)
delta_rel = abs(delta_f - delta_p) / max(abs(delta_p), 1e-30)

print(f"|SO4_final_F - SO4_final_P|             = {abs_err:.3e}")
print(f"|deltaSO4_F - deltaSO4_P| / deltaSO4_P  = {delta_rel:.3e}")

assert math.copysign(1, delta_f) == math.copysign(1, delta_p), \
    "Fortran and Python disagree on the sign of Delta SO4."
assert delta_rel < 1.0, (
    f"Delta SO4 differs by more than 100% between Fortran ({delta_f:.3e}) "
    f"and Python ({delta_p:.3e}); investigate solver/IC drift."
)

# Visual summary
fig, ax = plt.subplots(figsize=(7, 3.5))
labels = ["Fortran", "Python"]
deltas = np.array([delta_f, delta_p])
bars = ax.barh(labels, deltas, color=["#2E86AB", "#E07A5F"])
ax.set_xlabel(r"$\Delta$ SO$_4^{2-}$ over 1800 s  (mol m$^{-3}$)")
ax.set_title("MIAM Fortran vs Python: kinetic SO$_4$ production")
ax.grid(axis="x", alpha=0.3)
for bar, val in zip(bars, deltas):
    ax.text(val, bar.get_y() + bar.get_height() / 2, f"  {val:.3e}",
            va="center", fontsize=9)
plt.tight_layout()
plt.show()

print("\nPhase 7 passed: Fortran MIAM bindings produce SO4 in agreement with Python.")